# 电影圈数据

In [6]:
import pandas as pd
import numpy as np
import networkx as nx
import scipy
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Heiti TC']
plt.rcParams['axes.unicode_minus'] = False
from networkx.algorithms import bipartite

In [7]:
import sys
sys.path.append("..")

# 数据处理

## main

In [8]:
def works_data_process(df_raw):
    df_movie = df_raw.loc[(df_raw['k_region'] != '其他合拍')].copy()
    # k_cast_id转为str
    df_movie['k_cast_id'] = df_movie['k_cast_id'].apply(lambda x: str(x))
    # 仅保留演员，导演，编剧数据
    df_movie = df_movie[df_movie['k_role'].isin(['演员', '导演', '编剧'])]
    df_movie = df_movie.reset_index(drop=True).copy(deep=True)
    # 添加新的movie_id_m列
    df_movie['movie_id_m'] = df_movie['k_movie_id'].apply(
        lambda x: 'm' + str(x))
    # 合并影人职责
    df_cast = df_movie[['k_cast_id', 'cast_name', 'k_role']].drop_duplicates()
    df_cast_agg = df_cast.groupby([
        'k_cast_id', 'cast_name'
    ])['k_role'].apply(lambda x: '/'.join(sorted(x.unique()))).reset_index()
    df_movie['cast_role_agg'] = df_movie['k_cast_id'].map(
        df_cast_agg.set_index('k_cast_id')['k_role'])
    # 每部电影的主要演员，is_main_cast==‘是’， k_role==‘演员’, index小于=5
    # 为每行添加一个main_cast列，按k_movie_id分组，按is_main_cast==‘是’，k_role=='演员'降序，index升序排序后，取前5个演员的cast_name连接起来
    # 1. 筛选并排序：只保留“是演员”且“是主演”的行，按 index 升序
    cast_sorted = df_movie[(df_movie['is_main_cast'] == '是')
                           & (df_movie['k_role'] == '演员')].sort_values(
                               by=['k_movie_id', 'index'],
                               ascending=[True, True])

    # 2. 分组聚合：取每个电影前5名，用逗号连接
    main_cast_series = cast_sorted.groupby('k_movie_id')['cast_name'].apply(
        lambda x: ', '.join(x.head(5)))

    # 3. 映射回原表：使用 map 匹配 k_movie_id，确保索引正确
    df_movie['main_cast'] = df_movie['k_movie_id'].map(main_cast_series)

    # 将int64类型的列转换为普通int类型
    # int64_cols = df_movie.select_dtypes(include=['int64']).columns
    # for col in int64_cols:
    #     df_movie[col] = df_movie[col].astype(int)
    return df_movie


## 整体数据

In [10]:
df_raw = pd.read_csv("dwd_cast_works.csv")
df_movie = works_data_process(df_raw)
df_movie_rated = df_movie[df_movie['is_rating'] == '有评分'].copy()
df_movie_rated = df_movie_rated.reset_index(drop=True).copy(deep=True)
df_movie.to_csv("dwd_cast_works_processed.csv", index=False)
df_movie_rated.to_csv("dwd_cast_works_rated_processed.csv", index=False)

## 特定数据集

In [11]:
file_name = "zwyg"
df_raw_specific = pd.read_csv(f"{file_name}.csv")
df_movie_specific = works_data_process(df_raw_specific)
df_movie_rated_specific = df_movie_specific[df_movie_specific['is_rating'] == '有评分'].copy()
df_movie_rated_specific = df_movie_rated_specific.reset_index(drop=True).copy(deep=True)
df_movie_specific.to_csv(f"{file_name}_processed.csv", index=False)
df_movie_rated_specific.to_csv(f"{file_name}_rated_processed.csv", index=False)

# 数据统计

## 年度数据

In [ ]:
# 按年统计每年的电影数量
df_count_by_year = df_movie_rated.groupby('k_movie_year')['movie_id_m'].nunique().reset_index()
df_count_by_year.columns = ['year', 'movie_count']
df_count_by_year = df_count_by_year.sort_values('year')
df_count_by_year

In [ ]:
# 绘图
plt.figure(figsize=(10, 6))
plt.plot(df_count_by_year['year'], df_count_by_year['movie_count'], marker='o')
plt.title('年度电影数量统计')
plt.xlabel('年份')
plt.ylabel('电影数量')
plt.xticks(df_count_by_year['year'], rotation=45)
# plt.tight_layout()
plt.show()